In [0]:
%sql

CREATE TABLE IF NOT EXISTS databricksbootcamp.aerosurf.asset_status (

    asset_sk BIGINT GENERATED ALWAYS AS IDENTITY,

    asset_id STRING,
    beach STRING,
    category STRING,
    condition STRING,
    hire_status STRING,
    inspected_ts TIMESTAMP,

    defect_notes STRING,

    start_ts TIMESTAMP,
    end_ts TIMESTAMP,
    is_current BOOLEAN

)
USING DELTA

TBLPROPERTIES (
    delta.enableChangeDataFeed = true
);

In [0]:

BRONZE_TABLE = "databricksbootcamp.aerosurf.raw_inspections"
SILVER_TABLE = "databricksbootcamp.aerosurf.asset_status"

# Check Bronze schema
bronze_columns = spark.table(BRONZE_TABLE).columns

# Add defect_notes if it has not arrived yet
if "defect_notes" not in bronze_columns:

    spark.sql(f"""
        ALTER TABLE {BRONZE_TABLE}
        ADD COLUMNS (defect_notes STRING)
    """)

# Check Silver schema
silver_columns = spark.table(SILVER_TABLE).columns

# Add defect_notes to Silver if missing
if "defect_notes" not in silver_columns:

    spark.sql(f"""
        ALTER TABLE {SILVER_TABLE}
        ADD COLUMNS (defect_notes STRING)
    """)

print("Schema validation completed.")

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW scd2_staged AS

WITH latest_source AS (

    SELECT
        asset_id,
        beach,
        category,
        condition,
        hire_status,
        inspected_ts,
        defect_notes,
        bronze_ingested_at

    FROM (

        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY asset_id
                ORDER BY
                    inspected_ts DESC NULLS LAST,
                    bronze_ingested_at DESC NULLS LAST
            ) AS rn

        FROM databricksbootcamp.aerosurf.raw_inspections

    )

    WHERE rn = 1
),

compared AS (

    SELECT
        s.*,

        t.asset_id AS existing_asset_id,

        CASE

            WHEN t.asset_id IS NULL THEN TRUE

            WHEN NOT (
                t.beach        <=> s.beach
                AND t.category     <=> s.category
                AND t.condition    <=> s.condition
                AND t.hire_status  <=> s.hire_status
                AND t.inspected_ts <=> s.inspected_ts
                AND t.defect_notes <=> s.defect_notes
            )
            THEN TRUE

            ELSE FALSE

        END AS has_changed

    FROM latest_source s

    LEFT JOIN databricksbootcamp.aerosurf.asset_status t

        ON s.asset_id = t.asset_id
        AND t.is_current = TRUE
),

staged AS (

    -- Expire existing current version
    SELECT
        asset_id AS merge_key,
        asset_id,
        beach,
        category,
        condition,
        hire_status,
        inspected_ts,
        defect_notes,
        bronze_ingested_at,
        'EXPIRE' AS action

    FROM compared

    WHERE existing_asset_id IS NOT NULL
      AND has_changed = TRUE

    UNION ALL

    -- Insert new version
    SELECT
        CAST(NULL AS STRING) AS merge_key,
        asset_id,
        beach,
        category,
        condition,
        hire_status,
        inspected_ts,
        defect_notes,
        bronze_ingested_at,
        'INSERT' AS action

    FROM compared

    WHERE existing_asset_id IS NULL
       OR has_changed = TRUE

)

SELECT * FROM staged;

In [0]:
%sql

MERGE INTO databricksbootcamp.aerosurf.asset_status AS t

USING scd2_staged AS s

ON
    t.asset_id = s.merge_key
    AND t.is_current = TRUE

-- Expire old version
WHEN MATCHED AND s.action = 'EXPIRE'

THEN UPDATE SET

    t.is_current = FALSE,

    t.end_ts = COALESCE(
        s.inspected_ts,
        s.bronze_ingested_at,
        CURRENT_TIMESTAMP()
    )

-- Insert new version
WHEN NOT MATCHED AND s.action = 'INSERT'

THEN INSERT (

    asset_id,
    beach,
    category,
    condition,
    hire_status,
    inspected_ts,
    defect_notes,
    start_ts,
    end_ts,
    is_current

)

VALUES (

    s.asset_id,
    s.beach,
    s.category,
    s.condition,
    s.hire_status,
    s.inspected_ts,
    s.defect_notes,

    COALESCE(
        s.inspected_ts,
        s.bronze_ingested_at,
        CURRENT_TIMESTAMP()
    ),

    NULL,
    TRUE

);


In [0]:
%sql

-- Complete SCD Type 2 history

SELECT *
FROM databricksbootcamp.aerosurf.asset_status
ORDER BY asset_id, start_ts;

In [0]:
%sql

-- Current records only

SELECT *
FROM databricksbootcamp.aerosurf.asset_status
WHERE is_current = TRUE;

In [0]:
%sql

-- Find duplicate current records

SELECT
    asset_id,
    COUNT(*) AS current_count

FROM databricksbootcamp.aerosurf.asset_status

WHERE is_current = TRUE

GROUP BY asset_id

HAVING COUNT(*) > 1;